In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
import plotly.express as px
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate


from utilsforecast.losses import *

from utilsforecast.losses import *

import plotly.io as pio

from utilsforecast.losses import *
from metrics_utils import winkler_score
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial
from sklearn.linear_model import RidgeCV
from utilsforecast.losses import *
from plotting_utils import (
    plot_data_availability_heatmap,
    plot_missing_percentage,
    plotly_series as plot_series,
)
from statsforecast.models import SklearnModel

from statsforecast.models import (
    SeasonalNaive,
    AutoETS,
    MSTL,
)
from xgboost import XGBRegressor

# Introduction to Probabilistic Forecasting

Traditional time series forecasting methods typically provide a single predicted value for each future time point. This is known as **point forecasting**. However, real-world data is often uncertain and subject to various sources of randomness. For example, predicting tomorrow's electricity consumption or next week's sales involves many unknown factors.

**Probabilistic forecasting** addresses this uncertainty by predicting a range of possible future values, along with their associated probabilities. Instead of answering "What is the most likely value?", probabilistic forecasting answers "What is the probability that the value will fall within a certain range?".

## Why Probabilistic Forecasts Matter

- **Quantifying Uncertainty:** Probabilistic forecasts provide a measure of confidence in predictions, which is crucial for risk management and decision-making.
- **Better Decision Support:** Businesses can plan for best-case, worst-case, and most-likely scenarios.
- **Real-World Relevance:** Many applications (e.g., energy demand, finance, weather) require understanding the full range of possible outcomes, not just the average.

## Key Concepts

- **Prediction Interval:** A range within which the future value is expected to fall with a certain probability (e.g., 95% prediction interval).
- **Forecast Distribution:** The full probability distribution of possible future values, not just a single point estimate.

For example, instead of predicting that tomorrow's energy consumption will be exactly 100 kWh, a probabilistic forecast might say:

> There is a 90% chance that tomorrow's energy consumption will be between 95 and 110 kWh.

Mathematically, if $y_{t+h}$ is the value we want to forecast at time $t+h$, a probabilistic forecast provides the conditional distribution $P(y_{t+h} \mid \text{past data})$.

In the next sections, we'll explore how to generate and interpret probabilistic forecasts using modern time series tools.

In [3]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

unique_id,ds,start_timestamp,frequency,y,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary,__index_level_0__
str,list[datetime[ns]],datetime[ns],str,list[f64],i64,str,str,str,str,list[str],list[f64],list[i64],list[f64],list[f64],list[f64],list[f64],list[f64],list[str],list[str],list[f64],list[str],i64
"""MAC000002""","[2012-10-13 00:00:00, 2012-10-13 00:30:00, … 2014-02-27 23:30:00]",2012-10-13 00:00:00,"""30min""","[0.263, 0.269, … 1.2180001]",24144,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.08, 13.08, … 14.03]","[186, 186, … 200]","[8.78, 8.78, … 3.93]","[6.28, 6.28, … 1.61]","[1007.7, 1007.7, … 1004.62]","[7.55, 7.55, … 1.42]","[2.28, 2.28, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.84, 0.84, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",0
"""MAC000246""","[2012-01-01 00:00:00, 2012-01-01 00:30:00, … 2014-02-27 23:30:00]",2012-01-01 00:00:00,"""30min""","[0.509, 0.317, … 0.223]",37872,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[12.99, 12.99, … 14.03]","[229, 229, … 200]","[12.12, 12.12, … 3.93]","[10.97, 10.97, … 1.61]","[1008.1, 1008.1, … 1004.62]","[12.12, 12.12, … 1.42]","[5.9, 5.9, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.93, 0.93, … 0.85]","[""Mostly Cloudy"", ""Mostly Cloudy"", … ""Clear""]",1
"""MAC000450""","[2012-03-23 00:00:00, 2012-03-23 00:30:00, … 2014-02-27 23:30:00]",2012-03-23 00:00:00,"""30min""","[1.337, 1.426, … null]",33936,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[3.19, 3.19, … 14.03]","[78, 78, … 200]","[8.76, 8.76, … 3.93]","[7.25, 7.25, … 1.61]","[1027.41, 1027.41, … 1004.62]","[7.59, 7.59, … 1.42]","[2.18, 2.18, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""fog"", ""fog"", … ""clear-night""]","[0.9, 0.9, … 0.85]","[""Foggy"", ""Foggy"", … ""Clear""]",2
"""MAC001074""","[2012-05-09 00:00:00, 2012-05-09 00:30:00, … 2014-02-27 23:30:00]",2012-05-09 00:00:00,"""30min""","[0.18, 0.086, … null]",31680,"""ToU""","""ACORN-""","""ACORN-""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[10.51, 10.51, … 14.03]","[215, 215, … 200]","[11.46, 11.46, … 3.93]","[10.23, 10.23, … 1.61]","[1007.39, 1007.39, … 1004.62]","[11.46, 11.46, … 1.42]","[2.35, 2.35, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.92, 0.92, … 0.85]","[""Partly Cloudy"", ""Partly Cloudy"", … ""Clear""]",3
"""MAC003223""","[2012-09-18 00:00:00, 2012-09-18 00:30:00, … 2014-02-27 23:30:00]",2012-09-18 00:00:00,"""30min""","[0.076, 0.079, … 0.38]",25344,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.44, 13.44, … 14.03]","[236, 236, … 200]","[14.06, 14.06, … 3.93]","[10.82, 10.82, … 1.61]","[1011.09, 1011.09, … 1004.62]","[14.06, 14.06, … 1.42]","[3.86, 3.86, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.81, 0.81, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",4


In [4]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [5]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select([time_, id_, target_])
    .explode([time_, target_])
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000050""",0.175
2012-01-01 00:30:00,"""MAC000050""",0.212
2012-01-01 01:00:00,"""MAC000050""",0.313
2012-01-01 01:30:00,"""MAC000050""",0.302
2012-01-01 02:00:00,"""MAC000050""",0.257


In [6]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000193""",0.368
2012-01-01 00:30:00,"""MAC000193""",0.386
2012-01-01 01:00:00,"""MAC000193""",0.17
2012-01-01 01:30:00,"""MAC000193""",0.021
2012-01-01 02:00:00,"""MAC000193""",0.038


In [7]:
from statsforecast.models import (
    SeasonalNaive,
    AutoETS,
    ARIMA,
    MSTL,
)

In [8]:
sf = StatsForecast(
    models=[
        SeasonalNaive(season_length=48),
        AutoETS(season_length=48, model="ANA"),
        ARIMA(
            order=(1, 0, 3),
            seasonal_order=(1, 1, 1),
            season_length=48,
        ),
        MSTL(season_length=[48, 336], trend_forecaster=AutoETS(model="ANN")),
    ],
    freq="30m",
)

y_hat = sf.cross_validation(
    df=data.select([id_, time_, target_]),
    h=48,
    step_size=1,
    n_windows=1,
    level=[80],
).drop("cutoff")
y_hat = y_hat.with_columns(pl.col(pl.Float64).clip(0))

/Users/h33662/Projects/self/london-smart-meters-time-series-forecast/.venv/lib/python3.12/site-packages/statsforecast/arima.py:634: RuntimeWarning: divide by zero encountered in matmul
  a = T @ a
/Users/h33662/Projects/self/london-smart-meters-time-series-forecast/.venv/lib/python3.12/site-packages/statsforecast/arima.py:634: RuntimeWarning: overflow encountered in matmul
  a = T @ a
/Users/h33662/Projects/self/london-smart-meters-time-series-forecast/.venv/lib/python3.12/site-packages/statsforecast/arima.py:634: RuntimeWarning: invalid value encountered in matmul
  a = T @ a
/Users/h33662/Projects/self/london-smart-meters-time-series-forecast/.venv/lib/python3.12/site-packages/statsforecast/arima.py:636: RuntimeWarning: divide by zero encountered in matmul
  mm = T @ P
/Users/h33662/Projects/self/london-smart-meters-time-series-forecast/.venv/lib/python3.12/site-packages/statsforecast/arima.py:636: RuntimeWarning: overflow encountered in matmul
  mm = T @ P
/Users/h33662/Projects/sel

## The Advantage of Statistical Models: Natural Prediction Intervals

One of the great strengths of classical statistical time series models is their ability to provide **prediction intervals** alongside point forecasts. Unlike many machine learning models, which often require extra steps or assumptions to estimate uncertainty, statistical models are built on probability theory. This means they naturally quantify the uncertainty in their predictions.

### Why Are Prediction Intervals Important?

- **Risk Awareness:** Instead of just a single "best guess," you get a range that likely contains the true future value.
- **Decision Making:** Knowing the range of possible outcomes helps in planning for best-case and worst-case scenarios.
- **Model Trust:** Transparent uncertainty estimates make forecasts more trustworthy and actionable.

## Prediction Intervals in Exponential Smoothing (ETS)

The **Exponential Smoothing State Space Model (ETS)** is a popular statistical model for time series forecasting. ETS models not only predict the expected future value but also estimate the uncertainty around that prediction.

When you use an ETS model (like `AutoETS` in `statsforecast`), you can directly request prediction intervals at your desired confidence level (e.g., 80%, 95%). The model computes these intervals based on the estimated variance of the forecast errors.

### How Are Prediction Intervals Computed in ETS?

The prediction interval is typically calculated as:

$$
\hat{y}_{t+h|t} \pm z_{\alpha/2} \cdot \sigma_{t+h|t}
$$

- $\hat{y}_{t+h|t}$: The point forecast for $h$ steps ahead.
- $z_{\alpha/2}$: The critical value from the standard normal distribution (e.g., $z_{0.025} \approx 1.96$ for a 95% interval).
- $\sigma_{t+h|t}$: The standard deviation of the forecast at horizon $h$.

### Formula for the Forecast Standard Deviation in Probabilistic Forecasts

The **forecast standard deviation** quantifies the expected spread of forecast errors. For ETS models, the standard deviation for the $h$-step-ahead forecast is:

$$
\sigma_{t+h|t} = \sqrt{\sigma^2 \cdot V(h)}
$$

- $\sigma^2$: The estimated variance of the model's residuals (errors).
- $V(h)$: A function that depends on the model structure and the forecast horizon $h$ (for simple models, $V(h)$ often increases with $h$).

Model | Forecast variance: $\sigma_h^2$
--- | ---
$(A, N, N)$ | $ \sigma_h^2 = \sigma^2(1 + \alpha^2(h-1)) $
$(A, A, N)$ | $ \sigma_h^2 = \sigma^2\left[1 + (h-1)\left\{\alpha^2 + \alpha\beta h + \frac{1}{6}\beta^2h(2h-1)\right\}\right] $
$(A, A_d, N)$ | $ \sigma_h^2 = \sigma^2\left[1 + \alpha^2(h-1) + \frac{\beta\phi h}{(1-\phi)^2}\left\{2\alpha(1 - \phi) + \beta\phi\right\} - \frac{\beta\phi(1-\phi^h)}{(1-\phi)^2(1-\phi^2)}\left\{2\alpha(1 - \phi^2) + \beta\phi(1 + 2\phi - \phi^h)\right\}\right] $
$(A, N, A)$ | $ \sigma_h^2 = \sigma^2[1 + \alpha^2(h-1) + \gamma k(2\alpha + \gamma)] $
$(A, A, A)$ | $ \sigma_h^2 = \sigma^2\left[1 + (h-1)\left\{\alpha^2 + \alpha\beta h + \frac{1}{6}\beta^2h(2h-1)\right\} + \gamma k\{2\alpha + \gamma + \beta m(k+1)\}\right] $
$(A, A_d, A)$ | $ \sigma_h^2 = \sigma^2\left[1 + \alpha^2(h-1) + \gamma k(2\alpha + \gamma) + \frac{\beta\phi h}{(1-\phi)^2}\left\{2\alpha(1 - \phi) + \beta\phi\right\} - \frac{\beta\phi(1-\phi^h)}{(1-\phi)^2(1-\phi^2)}\left\{2\alpha(1 - \phi^2) + \beta\phi(1 + 2\phi - \phi^h)\right\} + \frac{2\beta\gamma\phi}{(1-\phi)(1-\phi^m)}\left\{k(1 - \phi^m) - \phi^m(1 - \phi^{mk})\right\}\right] $


### Important Note on Prediction Intervals in `statsforecast`

When using the `statsforecast` library's ETS models (such as `AutoETS`), the way prediction intervals are computed depends on the complexity of the model:

- **For simple ETS models** (for example, those with only additive components and no damping or multiplicative seasonality), `statsforecast` uses the **analytical formulas** for forecast variance and prediction intervals, as shown above. These formulas are derived from probability theory and provide exact intervals based on the model's estimated parameters and residual variance.

- **For more complex ETS models**—especially those with multiplicative errors, multiplicative seasonality, or damping—**there are no closed-form analytical formulas** for the prediction intervals. In these cases, `statsforecast` automatically switches to a **simulation-based approach**:
    - The model generates many possible future sample paths by simulating the random components of the ETS process.
    - Prediction intervals are then estimated from the percentiles of these simulated future paths.

**In summary:**  
- For simple ETS models, `statsforecast` uses the exact mathematical formulas for prediction intervals.
- For complex ETS models, it uses simulation to ensure you still get reliable, data-driven intervals.

This hybrid approach ensures that you always receive meaningful prediction intervals, regardless of the model's complexity—making `statsforecast` both robust and practical for real-world forecasting tasks.

**In summary:**  
Statistical models like ETS make it straightforward to obtain both point forecasts and prediction intervals, giving you a complete probabilistic picture of the future. This is a key reason why these models remain popular and useful in practical forecasting tasks.

In [9]:
plot_series(data, y_hat, max_insample_length=200, models=["AutoETS"], level=[80])

## Probabilistic Forecasting and Prediction Intervals with ARIMA

Just like ETS models, **ARIMA (AutoRegressive Integrated Moving Average)** models are a cornerstone of classical time series forecasting. ARIMA models are especially valued for their ability to capture autocorrelation and trends in time series data. But how do they handle uncertainty and provide **probabilistic forecasts**?

### How ARIMA Models Quantify Uncertainty

When you use an ARIMA model to forecast future values, it doesn't just give you a single "best guess" (the point forecast). It also estimates how much uncertainty surrounds that forecast, allowing you to construct **prediction intervals**.

#### Why Is This Important?

- **Risk Management:** Knowing the range of likely future values helps you prepare for best- and worst-case scenarios.
- **Model Trust:** Transparent intervals make it easier to trust and interpret your forecasts.
- **Decision Support:** Businesses and policymakers can make more informed choices when they understand the uncertainty in predictions.

### How Are Prediction Intervals Computed in ARIMA?

ARIMA models are built on probability theory. When you forecast $h$ steps ahead, the model calculates not only the expected value $\hat{y}_{t+h|t}$, but also the **forecast variance** $\sigma^2_{t+h|t}$, which reflects the uncertainty at each forecast horizon.

The **prediction interval** at a given confidence level (e.g., 80% or 95%) is:

$$
\hat{y}_{t+h|t} \pm z_{\alpha/2} \cdot \sigma_{t+h|t}
$$

- $\hat{y}_{t+h|t}$: Point forecast for $h$ steps ahead.
- $z_{\alpha/2}$: Critical value from the standard normal distribution (e.g., $z_{0.025} \approx 1.96$ for a 95% interval).
- $\sigma_{t+h|t}$: Standard deviation of the $h$-step-ahead forecast error.

#### How Is the Forecast Variance Calculated?

For ARIMA models, the forecast variance increases as you look further into the future. This is because uncertainty accumulates with each step ahead. The exact formula for $\sigma^2_{t+h|t}$ depends on the ARIMA parameters, but the general principle is:

- **Short-term forecasts** (small $h$): Lower variance, narrower intervals.
- **Long-term forecasts** (large $h$): Higher variance, wider intervals.

For a simple ARIMA(0,0,0) (i.e., white noise), the variance is constant. For more complex ARIMA models, the variance is calculated recursively, taking into account the AR and MA terms.
#### Example Formulas for ARIMA Forecast Variance

The forecast variance in ARIMA models depends on the specific AR and MA parameters. Here are some illustrative examples:

- **White Noise (ARIMA(0,0,0)):**
    $$
    \operatorname{Var}(y_{t+h|t}) = \sigma^2
    $$
    The forecast variance is constant at all horizons.

- **Random Walk (ARIMA(0,1,0)):**
    $$
    \operatorname{Var}(y_{t+h|t}) = h \cdot \sigma^2
    $$
    The variance increases linearly with the forecast horizon $h$.

- **AR(1) Model (ARIMA(1,0,0)):**
    $$
    \operatorname{Var}(y_{t+h|t}) = \sigma^2 \left(1 + \phi^2 + \phi^4 + \cdots + \phi^{2(h-1)}\right)
    $$
    Or, more compactly:
    $$
    \operatorname{Var}(y_{t+h|t}) = \sigma^2 \frac{1 - \phi^{2h}}{1 - \phi^2}
    $$
    where $\phi$ is the AR(1) coefficient.

- **General ARIMA(p,d,q):**
    The forecast variance is computed recursively, taking into account the AR and MA coefficients. For most practical purposes, statistical software (like `statsforecast`) handles these calculations internally.

**Key Point:**  
The further ahead you forecast ($h$ increases), the larger the forecast variance becomes, reflecting growing uncertainty. The exact rate of increase depends on the ARIMA model structure.

### Simulation vs. Analytical Formulas

- **Analytical Approach:** For most ARIMA models, the forecast variance can be calculated exactly using mathematical formulas derived from the model's structure.
- **Simulation Approach:** For very complex models or when analytical solutions are difficult, simulation (generating many possible future paths) can be used to estimate prediction intervals.

The `statsforecast` library, like most statistical packages, uses the analytical formulas for ARIMA prediction intervals whenever possible, ensuring both speed and accuracy.

### Real-World Analogy

Imagine you're predicting the temperature for the next week. If you use an ARIMA model, it will give you a best guess for each day, but also a range (interval) that reflects how much the temperature could realistically vary, based on past patterns and randomness.

### Key Takeaways

- **ARIMA models provide both point forecasts and prediction intervals.**
- **Prediction intervals widen as the forecast horizon increases, reflecting growing uncertainty.**
- **These intervals are grounded in probability theory, making them reliable and interpretable.**

**In summary:**  
ARIMA models are powerful not just for their forecasting accuracy, but also for their ability to quantify uncertainty. By providing prediction intervals, they give you a full probabilistic picture of the future—essential for robust decision-making in any time series application.

In [10]:
plot_series(data, y_hat, max_insample_length=200, models=["ARIMA"], level=[80])

## Probabilistic Forecasting and Prediction Intervals with MSTL

Just as with ETS and ARIMA, the **MSTL (Multiple Seasonal-Trend decomposition using Loess)** model can provide not only point forecasts but also **prediction intervals**—a crucial feature for quantifying uncertainty in time series forecasting.

### What is MSTL?

MSTL is a modern extension of classical time series decomposition methods. It is designed to handle time series data with **multiple seasonal patterns** (for example, daily and weekly cycles in electricity usage). MSTL decomposes the series into trend, multiple seasonal components, and remainder (noise), then forecasts each component separately before recombining them.

### How Does MSTL Quantify Uncertainty?

When you use MSTL for forecasting (as implemented in `statsforecast`), the model provides prediction intervals by estimating the uncertainty in the remainder (noise) component and propagating it through the forecast horizon.

#### The General Approach

- **Decomposition:** The time series is split into trend, multiple seasonalities, and remainder.
- **Forecasting:** Each component is forecasted into the future, often using a simple model (like ETS or ARIMA) for the trend and seasonal parts.
- **Uncertainty Estimation:** The variability in the remainder (the part not explained by trend or seasonality) is used to estimate the forecast error variance.
- **Prediction Intervals:** The intervals are constructed by adding and subtracting a multiple of the estimated standard error from the point forecast, just as with ETS and ARIMA:

    $$
    \hat{y}_{t+h|t} \pm z_{\alpha/2} \cdot \sigma_{t+h|t}
    $$

    - $\hat{y}_{t+h|t}$: The MSTL point forecast for $h$ steps ahead.
    - $z_{\alpha/2}$: The critical value from the standard normal distribution (e.g., $z_{0.10} \approx 1.28$ for an 80% interval).
    - $\sigma_{t+h|t}$: The estimated standard deviation of the forecast error at horizon $h$.

#### Analytical Estimation of Forecast Variance in MSTL

For simple MSTL models—where the remainder (noise) is assumed to be white noise with constant variance—the forecast variance at horizon $h$ can be estimated analytically, much like in ETS models. The key idea is that, after removing trend and seasonal components, the uncertainty in the forecast comes primarily from the unexplained remainder.

If the remainder has variance $\sigma^2$, then the forecast variance for $h$ steps ahead is:

$$
\operatorname{Var}(\hat{y}_{t+h|t}) = \sigma^2 \cdot h
$$

- $\sigma^2$: The variance of the remainder (noise) component, estimated from the historical data.
- $h$: The forecast horizon (how many steps ahead you are predicting).

This formula assumes that the remainder is independent and identically distributed (i.i.d.) noise. As you forecast further into the future, the uncertainty accumulates linearly with $h$.

**Key Point:**  
- For short-term forecasts ($h$ small), the intervals are narrower.
- For long-term forecasts ($h$ large), the intervals widen, reflecting growing uncertainty.

In practice, `statsforecast` estimates $\sigma^2$ from the in-sample remainder and uses this formula to construct prediction intervals for MSTL when the assumptions hold. If the remainder is not white noise, simulation-based intervals are preferred.

#### Analytical vs. Simulation-Based Intervals

- **Analytical Intervals:** For simple MSTL setups (where the remainder is assumed to be white noise), the forecast variance can be estimated analytically, similar to ETS.
- **Simulation-Based Intervals:** If the remainder shows complex patterns or non-constant variance, or if the trend/seasonal forecasts are uncertain, MSTL may use simulation (bootstrapping the remainder or simulating future paths) to estimate the prediction intervals.

### Key Points for Beginners

- **MSTL prediction intervals reflect the uncertainty in the unexplained part of the data (the remainder).**
- **If the remainder is highly variable, the intervals will be wider, indicating more uncertainty.**
- **If the model captures most of the structure (trend and seasonality), the intervals will be narrower, but only if the remainder is truly random.**

### Real-World Analogy

Imagine you’re forecasting electricity demand for a building. MSTL helps you account for both daily and weekly cycles (e.g., higher usage on weekdays, peaks in the evening). The prediction interval tells you, "Given all the patterns we’ve found, and the random fluctuations we couldn’t explain, here’s the range where we expect future demand to fall."

### Practical Note

In `statsforecast`, when you request prediction intervals from MSTL, the library automatically chooses the best method (analytical or simulation) based on the data and model complexity—ensuring you get reliable uncertainty estimates.

---

**In summary:**  
MSTL provides both point forecasts and prediction intervals by decomposing the time series, forecasting each component, and quantifying the uncertainty in the remainder. This makes MSTL a powerful tool for probabilistic forecasting when your data has multiple seasonal patterns.

In [11]:
plot_series(data, y_hat, max_insample_length=200, models=["MSTL"], level=[80])

## The Challenge of Negative Prediction Intervals in Statistical Models

When using statistical models like ETS or ARIMA for probabilistic forecasting, a common issue arises: **prediction intervals can sometimes extend below zero**, especially when forecasting time series with small or near-zero values (such as energy consumption, sales counts, or rainfall).

### Why Does This Happen?

- **Symmetric Intervals:** Most statistical models assume that forecast errors are symmetrically distributed around the point forecast (often assuming normality).
- **No Lower Bound:** The analytical formulas for prediction intervals do not "know" that the true values cannot be negative. As a result, when the point forecast is small and the estimated uncertainty (standard deviation) is relatively large, the lower bound of the interval can dip below zero.

**Example:**  
Suppose your model predicts $\hat{y}_{t+h|t} = 2$ (e.g., 2 kWh of energy), and the 80% prediction interval is $\pm 3$. The interval becomes $[2 - 3, 2 + 3] = [-1, 5]$, which includes negative values—even though negative energy consumption is impossible.

### Why Is This a Problem?

- **Interpretability:** Negative forecasts are not meaningful for quantities that are inherently non-negative.
- **Decision-Making:** Using such intervals can lead to confusion or poor decisions, especially in applications like inventory management or resource planning.

### How Can We Remedy This?

There are several practical approaches to address negative prediction intervals:

1. **Truncate at Zero:**  
    The simplest solution is to set any negative lower bounds to zero:
    $$
    \text{Lower Bound} = \max(0, \hat{y}_{t+h|t} - z_{\alpha/2} \cdot \sigma_{t+h|t})
    $$
    This ensures that the interval never goes below zero, but it can distort the true coverage probability (the interval may now contain the true value less often than intended).

2. **Transform the Data:**  
    - **Log Transformation:** Apply a logarithmic transformation to the data before modeling:
      $$
      y' = \log(y + c)
      $$
      where $c$ is a small constant to handle zeros.
    - Model and forecast in the log scale, then **exponentiate** the forecasts and intervals to return to the original scale. This naturally enforces non-negativity:
      $$
      \hat{y} = \exp(\hat{y}')
      $$
    - Note: Prediction intervals must be carefully back-transformed, as the log transformation changes the distribution of errors.

3. **Use Models Designed for Non-Negative Data:**  
    Some statistical models (e.g., Poisson or Gamma regression) are specifically designed for count or strictly positive data. These models inherently produce non-negative forecasts and intervals.

4. **Quantile Forecasting with ML Models:**  
    Machine learning models trained with quantile loss (see next section) can be constrained to predict only non-negative quantiles, providing more realistic intervals for non-negative data.

### Key Takeaway

**Always check your prediction intervals for plausibility!**  
If your data cannot be negative, ensure your intervals respect this constraint—either by truncating, transforming, or choosing appropriate models. This will make your probabilistic forecasts both more interpretable and more useful for real-world decision-making.

## The Pitfall of Too-Narrow Prediction Intervals: Unaccounted Uncertainty in Time Series Forecasting

Prediction intervals are a powerful feature of probabilistic forecasting—they give us a range within which we expect future values to fall, with a certain probability (e.g., 80% or 95%). However, these intervals are only as reliable as the assumptions and information built into the model. A common issue, especially for beginners, is that prediction intervals can be **too narrow**—meaning they underestimate the true uncertainty of the future.

### Why Do Prediction Intervals Become Too Narrow?

Prediction intervals are calculated based on the model's understanding of uncertainty. If the model doesn't account for all sources of variability, the intervals will be misleadingly tight.

There are at least four sources of uncertainty in forecasting using time series models:

- **The random error term:** Every time series model includes a random error (or noise) component, representing unpredictable fluctuations that cannot be explained by the model. This is the inherent randomness present in any real-world process.
- **The parameter estimates:** When fitting a model to historical data, we estimate parameters (like means, variances, or coefficients). These estimates are themselves uncertain, especially with limited or noisy data, and this uncertainty carries over into the forecasts.
- **The choice of model for the historical data:** Different models may fit the same data in different ways. The process of selecting a model introduces uncertainty, as the chosen model may not perfectly capture the true underlying process.
- **The continuation of the historical data generating process into the future:** Forecasts assume that the future will behave similarly to the past (i.e., the data generating process remains stable). However, structural changes, regime shifts, or unforeseen events can cause this assumption to break down, adding another layer of uncertainty.

### Real-World Analogy

Imagine you're forecasting the number of customers in a coffee shop for tomorrow. Let's see how each of the four sources of uncertainty can affect your prediction intervals:

- **Random error term:** Even if you know everything about your shop, there will always be unpredictable events—maybe someone brings a large group unexpectedly, or a regular skips their daily visit. This randomness is like the "noise" in your data.
- **Parameter estimates:** Suppose you estimate the average number of customers based on the past month. If your data is limited or especially variable, your estimate of the "true" average is itself uncertain. Maybe you think you'll get 50 customers, but the real average could be 45 or 55.
- **Choice of model:** You might use a simple average, or you might try to account for trends (like more customers on weekends). If you pick the wrong model—say, you ignore a growing trend or seasonality—your forecasts (and their intervals) will be off.
- **Future changes in the data-generating process:** Perhaps a new competitor opens nearby, or there's a sudden change in weather patterns. If your model assumes the future will be just like the past, it won't capture these shifts, making your intervals too optimistic.

If you ignore any of these sources, your prediction interval might say, "We're 95% sure we'll get between 48 and 52 customers tomorrow." But in reality, because of all these uncertainties, the true number could easily fall outside that range. That's why it's so important to account for all sources of uncertainty when building and interpreting prediction intervals!

### Mathematical Perspective

Recall the formula for a prediction interval:

$$
\hat{y}_{t+h|t} \pm z_{\alpha/2} \cdot \text{SE}(\hat{y}_{t+h|t})
$$

If the standard error $\text{SE}(\hat{y}_{t+h|t})$ is underestimated (because the model doesn't "see" all sources of uncertainty), the interval will be too small.

### Consequences of Too-Narrow Intervals

- **Overconfidence:** Decision-makers may believe forecasts are more certain than they really are.
- **Missed Risks:** Unexpected events or variability can lead to costly surprises.
- **Poor Planning:** Inventory, staffing, or resource allocation may be insufficient for real-world fluctuations.

### How to Address This Problem

- **Model Diagnostics:** Always check how often actual values fall inside your prediction intervals. If it's much less than the nominal level (e.g., only 60% of values inside a 95% interval), your intervals are too narrow.
- **Include More Sources of Uncertainty:** Add relevant features, use models that allow for changing variance, or combine multiple models.
- **Use Robust Models:** Some advanced models (like those in `nixtla`'s `neuralforecast` or `mlforecast`) can better capture complex uncertainty.

**In summary:**  
Prediction intervals are only as good as the uncertainty your model accounts for. Always be cautious of intervals that seem "too good to be true"—they probably are! Regularly validate your intervals against real outcomes and strive to include all relevant sources of uncertainty in your modeling process.

## The Second Pitfall: Assumptions About Residuals in Statistical Models

When using classical statistical models like ETS or ARIMA for probabilistic forecasting, another common pitfall is making strong assumptions about the **residuals** (the differences between the observed values and the model's predictions).

### What Are Residuals?

- **Residuals** are the "leftover" part of the data after the model has explained as much as it can. Mathematically, for each time point $t$, the residual is $e_t = y_t - \hat{y}_t$.

### The Standard Assumptions

Most statistical models assume that residuals are:

- **Uncorrelated:** Each residual is independent of the others (no pattern or structure left).
- **Normally distributed:** Residuals follow a bell-shaped curve, centered at zero.

These assumptions are crucial because:

- **Prediction intervals** are calculated using the estimated variance of the residuals.
- The formulas for forecast uncertainty (and thus the width of prediction intervals) rely on these properties.

### Why Is This a Pitfall?

In real-world time series, these assumptions often **do not hold**:

- **Residuals may be autocorrelated:** If there is leftover structure (e.g., seasonality, trends, or cycles) not captured by the model, residuals will be correlated. This means the model is missing something important.
- **Residuals may not be normal:** Outliers, skewness, or heavy tails are common in real data, violating the normality assumption.

### Consequences

- **Prediction intervals may be misleading:** If residuals are autocorrelated or not normal, the calculated intervals can be too narrow or too wide, and the actual coverage (the percentage of true values falling inside the interval) will not match the nominal level (e.g., 80% or 95%).
- **Overconfidence or underconfidence:** You might trust your model too much (or too little), leading to poor decisions.

### How to Check and Address This

- **Diagnostic plots:** Always plot your residuals and check for patterns (autocorrelation plots, histograms, Q-Q plots).
- **Model refinement:** If residuals are autocorrelated, try a more complex model or add missing features.
- **Robust methods:** Consider models or interval estimation techniques that are less sensitive to these assumptions.

### Key Takeaway

**Never blindly trust the prediction intervals from statistical models without checking the residuals!**  
If the assumptions about residuals are violated, your uncertainty estimates may be unreliable. Always validate your model's assumptions and adjust your approach as needed to ensure trustworthy probabilistic forecasts.

In [12]:
from functools import partial

metrics = [
    mqloss,
    winkler_score,
    coverage,
]
evaluate(
    y_hat,
    metrics=metrics,
    level=[80],
)

unique_id,metric,SeasonalNaive,AutoETS,ARIMA,MSTL
str,str,f64,f64,f64,f64
"""MAC000193""","""mqloss""",0.103812,0.067623,0.063909,0.073659
"""MAC000193""","""winkler_score_level80""",2.076246,1.352466,1.27817,1.473174
"""MAC000193""","""coverage_level80""",0.75,0.791667,0.770833,0.8125


# Introduction to Quantile Forecasting and Quantile Loss

## What is Quantile Forecasting?

Quantile forecasting is a powerful approach in probabilistic time series forecasting. Instead of predicting just the average (mean) or a single value for each future time point, quantile forecasting predicts specific **quantiles** of the future value's distribution.

- **Quantile:** A quantile is a value below which a certain percentage of the data falls. For example, the 0.5 quantile (also called the median) is the value below which 50% of the data lies. The 0.9 quantile is the value below which 90% of the data lies.

**In quantile forecasting, you might predict:**
- The 0.1 quantile (10th percentile): "There is a 10% chance the value will be below this."
- The 0.5 quantile (median): "There is a 50% chance the value will be below this."
- The 0.9 quantile (90th percentile): "There is a 90% chance the value will be below this."

This gives a much richer picture of possible future outcomes, allowing you to understand not just the "most likely" value, but also the range and likelihood of extreme events.

## What is Quantile Loss?

To train models to predict quantiles, we use a special loss function called **quantile loss** (also known as "pinball loss"). This loss function penalizes the model differently depending on whether the prediction is above or below the actual value, and by how much.

The quantile loss for a quantile $\tau$ is defined as:

$$
L_\tau(y, \hat{y}) = 
\begin{cases}
\tau \cdot (y - \hat{y}) & \text{if } y \geq \hat{y} \\
(1 - \tau) \cdot (\hat{y} - y) & \text{if } y < \hat{y}
\end{cases}
$$

- $y$: The true value.
- $\hat{y}$: The predicted quantile.
- $\tau$: The quantile level (e.g., 0.1, 0.5, 0.9).

**Intuition:**  
- If you predict too low for a high quantile (e.g., 0.9), you get penalized more.
- If you predict too high for a low quantile (e.g., 0.1), you get penalized more.

This encourages the model to make predictions that match the desired quantile of the distribution.

## How is Quantile Forecasting Different from Prediction Intervals in Statistical Models?

### Prediction Intervals (Statistical Models)

- **Statistical models** (like ETS or ARIMA) typically provide **prediction intervals**: a lower and upper bound such that, for example, "there is an 80% chance the true value will fall between these bounds."
- These intervals are usually **symmetric** around the point forecast and are based on assumptions about the distribution of errors (often normality).
- The intervals are calculated using analytical formulas or simulation, as described in previous sections.

### Quantile Forecasts

- **Quantile forecasts** directly estimate specific quantiles (e.g., 0.1, 0.5, 0.9) of the future value's distribution.
- You can construct an interval by taking, for example, the 0.1 and 0.9 quantile forecasts, but the interval does not have to be symmetric.
- Quantile forecasts do **not** assume a specific error distribution—they are more flexible and robust to outliers or skewed data.

### Key Differences

| Feature                | Prediction Interval (Stat Models) | Quantile Forecasting        |
|------------------------|-----------------------------------|-----------------------------|
| Output                 | Lower & upper bounds (interval)   | Any quantile(s) you choose  |
| Symmetry               | Usually symmetric                 | Can be asymmetric           |
| Assumptions            | Often normality, constant variance| No distributional assumption|
| How computed           | Analytical/simulation formulas    | Directly by model           |
| Typical use            | Statistical models (ETS, ARIMA)   | ML models, deep learning    |

## Is Quantile Forecasting More for Machine Learning Models?

**Yes, quantile forecasting is especially popular in machine learning (ML) and deep learning models.**

- **Why?** ML models (like gradient boosting, random forests, neural networks) do not naturally provide prediction intervals. But they can be trained to predict quantiles using quantile loss.
- **Flexibility:** ML models can learn complex, nonlinear relationships and can estimate any quantile directly, making them suitable for data with non-normal, skewed, or heteroscedastic (changing variance) distributions.
- **Examples:** Libraries like `mlforecast` and `neuralforecast` from Nixtla, as well as scikit-learn and XGBoost, support quantile regression.

**In summary:**  
- **Prediction intervals** from statistical models are based on analytical or simulation methods and rely on distributional assumptions.
- **Quantile forecasts** are more flexible, can be asymmetric, and are especially useful with ML models, which can be trained to predict any quantile directly using quantile loss.

This makes quantile forecasting a powerful tool for modern, data-driven time series analysis—especially when you want to capture the full range of possible future outcomes without strong assumptions about the data.

In [13]:
from mlforecast.lag_transforms import (
    RollingStd,
    SeasonalRollingMean,
    SeasonalRollingStd,
    ExponentiallyWeightedMean,
)

lags = [1, 2, 48, 336]
lag_transforms = {
    1: [
        RollingMean(window_size=3),
        RollingMean(window_size=6),
        RollingMean(window_size=12),
        RollingMean(window_size=48),
        RollingStd(window_size=3),
        RollingStd(window_size=6),
        RollingStd(window_size=12),
        RollingStd(window_size=48),
        ExponentiallyWeightedMean(alpha=0.25),
    ],
    48: [
        RollingMean(window_size=7),
        RollingMean(window_size=14),
        RollingStd(window_size=7),
        RollingStd(window_size=14),
        SeasonalRollingMean(season_length=48, window_size=3),
        SeasonalRollingStd(season_length=48, window_size=3),
    ],
    336: [
        RollingMean(window_size=4),
        RollingMean(window_size=8),
        RollingStd(window_size=4),
        RollingStd(window_size=8),
        SeasonalRollingMean(season_length=336, window_size=3),
        SeasonalRollingStd(season_length=336, window_size=3),
    ],
}

In [14]:
features = [
    partial(
        fourier, season_length=2 * 24, k=10
    ),  # Daily seasonality (48 observations per day)
    partial(
        fourier, season_length=2 * 24 * 7, k=5
    ),  # Weekly seasonality (336 observations per week)
    partial(
        fourier, season_length=2 * 24 * 365, k=3
    ),  # Annual seasonality (approx. 17520 observations per year)
]
data_fourier, data_futr_fourier = pipeline(
    data.select([id_, time_, target_]),
    features=features,
    freq="30m",
    h=48,  # Horizon for future features
)

mlf = MLForecast(
    models=[],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

In [15]:
quantiles = [0.1, 0.5, 0.9]


def get_quantile_xgb(q):
    return XGBRegressor(
        objective="reg:quantileerror",  # For XGBoost >= 1.7.0, use 'reg:quantileerror'
        quantile_alpha=q,
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        verbosity=0,
    )


models = [
    SklearnModel(get_quantile_xgb(quantiles[0]), alias="XGBRegressor-lo-80"),
    SklearnModel(get_quantile_xgb(quantiles[1]), alias="XGBRegressor"),
    SklearnModel(get_quantile_xgb(quantiles[2]), alias="XGBRegressor-hi-80"),
]

sf = StatsForecast(
    models=models,
    freq="30m",
)

y_hat = sf.cross_validation(
    df=mlf.preprocess(data_fourier, static_features=[]),
    h=48,
    step_size=1,
    n_windows=1,
).drop("cutoff")

In [16]:
plot_series(data, y_hat, max_insample_length=200, models=["XGBRegressor"], level=[80])

In [17]:
evaluate(
    y_hat,
    metrics=metrics,
    level=[80],
)

unique_id,metric,XGBRegressor
str,str,f64
"""MAC000193""","""mqloss""",0.052866
"""MAC000193""","""winkler_score_level80""",1.057324
"""MAC000193""","""coverage_level80""",0.6875
